In [1]:

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor

sys.path.append('../')
sys.path.append('./')
import importlib
import yaml
import torch

from tqdm.auto import tqdm
import logging

# logging.getLogger('sox').setLevel(logging.ERROR)
# logger = logging.getLogger('sox')
# logger.setLevel('CRITICAL')


In [ ]:
import lightning_scripts.lightning_ssl_matched_speech_in_noise as lightning 
importlib.reload(lightning)


LitAudioSSL = lightning.LitAudioSSL

## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/resnet18_barlow_invariant_only_lmbda_1e-2_lr_2e-1_w_invar_augment_no_avgpool.yaml"
# config_path = "model_configs/barlow_word_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_invar_augment.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 4
config['hparas']['batch_size'] = 64
config['hparas']['global_batch_size'] = 64
config['num_gpus'] = 1 

model = LitAudioSSL(config)
model(torch.randn(4, 1, 40000))


torch.Size([4, 4096]) Linear(in_features=4096, out_features=512, bias=False)
torch.Size([4, 512]) BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
torch.Size([4, 512]) ReLU()
torch.Size([4, 512]) Linear(in_features=512, out_features=512, bias=False)


(tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0878, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0670, 0.0017],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0643, 0.0039],
         [0.0000, 0.0034, 0.0000,  ..., 0.0000, 0.0641, 0.0000]],
        grad_fn=<ReluBackward0>),
 tensor([[ 0.0492, -0.1979,  0.3314,  ..., -0.2375, -0.2312,  0.1309],
         [ 0.3151,  0.0366,  0.1380,  ..., -0.0729,  0.2615,  0.1426],
         [-0.2371,  0.1742,  0.2815,  ...,  0.1615, -0.0869, -0.2350],
         [-0.2276,  0.2638,  0.1391,  ...,  0.2307,  0.2306,  0.0670]],
        grad_fn=<MmBackward0>),
 {'signal/word_int': tensor([[ 0.0068, -0.0032,  0.0059,  ...,  0.0059, -0.0069,  0.0180],
          [ 0.0030, -0.0026,  0.0074,  ...,  0.0004, -0.0088,  0.0152],
          [ 0.0027, -0.0003,  0.0050,  ..., -0.0010, -0.0116,  0.0142],
          [ 0.0038, -0.0005,  0.0066,  ..., -0.0003, -0.0103,  0.0131]],
         grad_fn=<AddmmBackward0>),
  'signal/speaker_int': tensor([[-0.0283,  0

ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 512])

In [3]:
config['data'].get("skip_aug_match", False)


True

In [8]:
from lightning_scripts import jsinV3DataLoader_precombined_batched 

importlib.reload(jsinV3DataLoader_precombined_batched)
MatchedSpeechInNoiseDatasetBatched = jsinV3DataLoader_precombined_batched.MatchedSpeechInNoiseDatasetBatched

dataset = MatchedSpeechInNoiseDatasetBatched(speech_h5_path=config['data']['val_speech_h5_path'],
                                                     noise_h5_path=config['data']['val_noise_h5_path'],
                                                     low_db=config['audio_transforms']['low_snr'],
                                                     high_db=config['audio_transforms']['high_snr'],
                                                     db_spl=config['audio_transforms']['dbspl'],
                                                     batch_size=config['hparas']['batch_size'],
                                                     signal_augment=config['data'].get("signal_augment", False),
                                                     target_keys=config['data'].get("target_keys", None),
                                                     skip_aug_match=config['data'].get("skip_aug_match", False),

                                                     )
[eg11, eg12, eg21, eg22], labels = dataset[0]

In [13]:
eg_ix = 10 
display(Audio(eg11[eg_ix], rate=20_000))
display(Audio(eg21[eg_ix], rate=20_000))

In [16]:
import pandas as pd
pd.read_pickle('train_config_manifests/barlow_equivariant_lmbda_search_kell2018_invar_to_augments.pkl')

{0: '/mnt/ceph/users/igriffith/projects/cochdnn/model_configs/equi_lmbda_search/barlow_dualtask_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment_eq_lmbda_5e-01_invar_augment.yaml',
 1: '/mnt/ceph/users/igriffith/projects/cochdnn/model_configs/equi_lmbda_search/barlow_dualtask_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment_eq_lmbda_1e-03_invar_augment.yaml',
 2: '/mnt/ceph/users/igriffith/projects/cochdnn/model_configs/equi_lmbda_search/barlow_dualtask_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment_eq_lmbda_4e-01_invar_augment.yaml',
 3: '/mnt/ceph/users/igriffith/projects/cochdnn/model_configs/equi_lmbda_search/barlow_dualtask_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment_eq_lmbda_1e-1_invar_augment.yaml',
 4: '/mnt/ceph/users/igriffith/projects/cochdnn/model_configs/equi_lmbda_search/barlow_dualtask_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment_eq_lmbda_2e-01_invar_augment.yaml',
 5: '/

In [6]:
batch = dataset[0]

In [8]:
batch[0][0]

tensor([[ 5.4993e-03,  1.6702e-03, -1.0030e-02,  ..., -3.2607e-03,
         -1.3480e-02,  5.6814e-03],
        [-3.9527e-03, -5.4219e-03, -7.3263e-03,  ..., -4.8766e-02,
         -4.0447e-02, -2.2994e-02],
        [ 1.8882e-02,  2.8544e-02,  3.5556e-02,  ..., -1.8424e-03,
         -2.2241e-03, -4.0600e-03],
        ...,
        [ 2.3297e-05,  1.3476e-03,  3.9609e-03,  ..., -4.1165e-04,
         -3.9116e-03, -4.6539e-03],
        [ 1.2179e-02,  1.2813e-02,  1.0281e-02,  ..., -1.8667e-02,
         -2.1732e-02, -2.2851e-02],
        [-1.9480e-02, -1.8370e-02, -1.6020e-02,  ...,  5.9041e-03,
          2.5728e-03,  2.4078e-03]])

In [1]:
model

NameError: name 'model' is not defined

In [4]:
trainer = L.Trainer(
                    # callbacks=[lr_monitor],
                    # limit_train_batches=5,
                    limit_val_batches=2,
                    max_epochs=5,
                    # callbacks=callbacks,
                    #  strategy='ddp_notebook',
                    #  reload_dataloaders_every_n_epochs=-1,
                    devices=1)

trainer.fit(model)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name            | Type                       | Params | Mode 
-----------------------------------------------------------------------
0 | audio_rep       | AudioToAudioRepresentation | 0      | train
1 | model           | ModelWithFrontEnd          | 116 M  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Rank 0 N training batches 317
This is the first step after restoring from a checkpoint!


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined